# Algorithmic Thinking: State, Structure, and Cost

This is the first notebook in the course. It gives you the habits used everywhere else: trace state, choose the right data shape, and estimate how the work grows.

The opening example reaches back to Eratosthenes, the Greek mathematician who used a sieve-like idea to find primes more than two thousand years ago. The same habit still drives modern computing: keep a state, apply a rule, and watch what changes.

You will start with one real algorithm, then build the Python vocabulary needed for graphs, matrices, queues, heaps, simulations, and complexity.

<details>
<summary>Notebook rhythm</summary>

Run a code cell, read the state it prints, then change one small input. Most algorithms become understandable when you can see what changed.

</details>

## 1. First Trace: Sieve of Eratosthenes

A trace is a record of state changing under a rule. The Sieve of Eratosthenes finds primes by keeping a list of possibilities and crossing off multiples.

Watch for three ingredients that return in later notebooks:

- state: `is_prime`
- rule: mark multiples as not prime
- stopping idea: no unmarked composite can escape forever

<details>
<summary>Core idea</summary>

When `number` is still marked prime, every multiple of `number` can be marked composite. Starting at `number * number` skips work already handled by smaller primes.

</details>

In [1]:
limit = 30
is_prime = [True] * (limit + 1)
is_prime[0] = False
is_prime[1] = False

snapshots = []

for number in range(2, limit + 1):
    if not is_prime[number]:
        continue

    snapshots.append((number, "found prime", is_prime.copy()))

    for multiple in range(number * number, limit + 1, number):
        is_prime[multiple] = False

    snapshots.append((number, "marked multiples", is_prime.copy()))

primes = [number for number, prime in enumerate(is_prime) if prime]


def show_possible(state: list[bool]) -> str:
    return " ".join(f"{index:02d}" if prime else ".." for index, prime in enumerate(state))


print(f"Primes up to {limit}: {primes}")
print("\nTrace preview:")
for number, action, state in snapshots[:8]:
    print(f"{number:>2} | {action:<16} | {show_possible(state)}")

Primes up to 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

Trace preview:
 2 | found prime      | .. .. 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30
 2 | marked multiples | .. .. 02 03 .. 05 .. 07 .. 09 .. 11 .. 13 .. 15 .. 17 .. 19 .. 21 .. 23 .. 25 .. 27 .. 29 ..
 3 | found prime      | .. .. 02 03 .. 05 .. 07 .. 09 .. 11 .. 13 .. 15 .. 17 .. 19 .. 21 .. 23 .. 25 .. 27 .. 29 ..
 3 | marked multiples | .. .. 02 03 .. 05 .. 07 .. .. .. 11 .. 13 .. .. .. 17 .. 19 .. .. .. 23 .. 25 .. .. .. 29 ..
 5 | found prime      | .. .. 02 03 .. 05 .. 07 .. .. .. 11 .. 13 .. .. .. 17 .. 19 .. .. .. 23 .. 25 .. .. .. 29 ..
 5 | marked multiples | .. .. 02 03 .. 05 .. 07 .. .. .. 11 .. 13 .. .. .. 17 .. 19 .. .. .. 23 .. .. .. .. .. 29 ..
 7 | found prime      | .. .. 02 03 .. 05 .. 07 .. .. .. 11 .. 13 .. .. .. 17 .. 19 .. .. .. 23 .. .. .. .. .. 29 ..
 7 | marked multiples | .. .. 02 03 .. 05 .. 07 .. .. .. 11 .. 13 .. .. .. 17 .. 19 .. .. .. 23 .. .. .. .. .. 

## 2. Name the Things: Dataclasses

Algorithms are easier to read when the code names the objects in the problem. A `dataclass` is a small object with named fields and almost no boilerplate.

Use one when a concept has a few stable facts: a place, road, task, page, point, center, or trace step.

In [2]:
from dataclasses import dataclass


@dataclass(frozen=True, order=True)
class Place:
    name: str

    def __str__(self) -> str:
        return self.name


@dataclass
class Road:
    start: Place
    end: Place
    cost: int


arcade = Place("Arcade")
library = Place("Library")
road = Road(arcade, library, 7)

print(road)
print("Start:", road.start)
print("End  :", road.end)
print("Cost :", road.cost)

Road(start=Place(name='Arcade'), end=Place(name='Library'), cost=7)
Start: Arcade
End  : Library
Cost : 7


## 3. Choose a Data Shape

Most course notebooks use the same small toolbox. Choose by the operation you need most often.

| Need | Usual structure |
| --- | --- |
| keep order | `list` |
| look up by name | `dict` |
| test membership | `set` |
| store a fixed pair | `tuple` |
| process first-in, first-out work | `deque` |
| process cheapest-first work | `heapq` |

In [3]:
from collections import deque
from heapq import heappop, heappush

path = ["Arcade", "Cafe"]
path.append("Library")

best_cost = {"Arcade": 0, "Cafe": 5, "Library": 8}
visited = {"Arcade", "Cafe"}
coordinate = (2, 3)

ready = deque(["Sketch", "Buy Supplies"])
next_ready = ready.popleft()

frontier = []
heappush(frontier, (8, "Library"))
heappush(frontier, (3, "Cafe"))
heappush(frontier, (5, "Diner"))
cheapest = heappop(frontier)

print("path:", path)
print("best cost to Library:", best_cost["Library"])
print("Cafe visited:", "Cafe" in visited)
print("coordinate:", coordinate)
print("next ready task:", next_ready)
print("cheapest frontier item:", cheapest)

path: ['Arcade', 'Cafe', 'Library']
best cost to Library: 8
Cafe visited: True
coordinate: (2, 3)
next ready task: Sketch
cheapest frontier item: (3, 'Cafe')


## 4. Graphs: Choices With Costs

A graph is a set of things plus connections between them. In code, the most common shape is an adjacency list: each node points to its outgoing neighbors.

This single pattern powers Dijkstra, A*, PageRank, Topological Sort, Floyd-Warshall, and network flow.

In [4]:
graph = {
    "Arcade": [("Cafe", 4), ("Library", 10)],
    "Cafe": [("Diner", 2)],
    "Diner": [("Library", 1)],
    "Library": [],
}

for place, neighbors in graph.items():
    choices = ", ".join(f"{neighbor} (cost {cost})" for neighbor, cost in neighbors)
    print(f"{place:>7}: {choices or 'no outgoing choices'}")

 Arcade: Cafe (cost 4), Library (cost 10)
   Cafe: Diner (cost 2)
  Diner: Library (cost 1)
Library: no outgoing choices


## 5. Matrices: Tables for Pairwise State

A matrix is a list of lists. Use one when every item can relate to every other item.

Read `matrix[row][col]` as: from the row item to the column item. Floyd-Warshall, PageRank, and dynamic programming all use this table-shaped thinking.

In [5]:
places = ["Arcade", "Cafe", "Library"]
index = {place: position for position, place in enumerate(places)}
INF = 999

matrix = [
    [0, 4, 10],
    [INF, 0, 2],
    [INF, INF, 0],
]

start = "Arcade"
end = "Library"
print("Cost Arcade -> Library:", matrix[index[start]][index[end]])

print("\nMatrix:")
print("        " + " ".join(f"{place:>7}" for place in places))
for place, row in zip(places, matrix):
    formatted = ["INF" if cost == INF else str(cost) for cost in row]
    print(f"{place:>7} " + " ".join(f"{cost:>7}" for cost in formatted))

Cost Arcade -> Library: 10

Matrix:
         Arcade    Cafe Library
 Arcade       0       4      10
   Cafe     INF       0       2
Library     INF     INF       0


## 6. Big-O: Growth, Not Stopwatch Time

Big-O describes how work grows as input size grows. It ignores constants and smaller terms so the dominant shape is visible.

Common shapes:

- `O(1)`: constant work
- `O(log n)`: repeatedly shrink the problem
- `O(n)`: one pass
- `O(n log n)`: sorting-style growth
- `O(n^2)`: pairs of items
- `O(n^3)`: triples of items, like Floyd-Warshall

In [6]:
def growth_counts(n: int) -> dict[str, int]:
    log_steps = 0
    size = n
    while size > 1:
        size //= 2
        log_steps += 1

    return {
        "O(1)": 1,
        "O(log n)": log_steps,
        "O(n)": n,
        "O(n log n)": n * log_steps,
        "O(n^2)": n * n,
        "O(n^3)": n * n * n,
    }

for n in [4, 8, 16, 32]:
    counts = growth_counts(n)
    print(f"n={n:>2} | " + " | ".join(f"{name}: {count:>5}" for name, count in counts.items()))

n= 4 | O(1):     1 | O(log n):     2 | O(n):     4 | O(n log n):     8 | O(n^2):    16 | O(n^3):    64
n= 8 | O(1):     1 | O(log n):     3 | O(n):     8 | O(n log n):    24 | O(n^2):    64 | O(n^3):   512
n=16 | O(1):     1 | O(log n):     4 | O(n):    16 | O(n log n):    64 | O(n^2):   256 | O(n^3):  4096
n=32 | O(1):     1 | O(log n):     5 | O(n):    32 | O(n log n):   160 | O(n^2):  1024 | O(n^3): 32768


## 7. Read Loops Like Shapes

For first-pass complexity analysis, count how many times the repeated work can happen.

Fast rules:

- one loop over `n` items is usually `O(n)`
- two nested loops over `n` items are usually `O(n^2)`
- three nested loops are usually `O(n^3)`
- a loop that halves the remaining work is usually `O(log n)`

The word "usually" matters: the changing input is what decides the final answer.

**Helper functions.** Define `count_one_loop`, `count_nested_loops`, `count_triple_loops` so the later cells stay focused on behavior.


In [ ]:
def count_one_loop(n: int) -> int:
    operations = 0
    for _ in range(n):
        operations += 1
    return operations

def count_nested_loops(n: int) -> int:
    operations = 0
    for _ in range(n):
        for _ in range(n):
            operations += 1
    return operations

def count_triple_loops(n: int) -> int:
    operations = 0
    for _ in range(n):
        for _ in range(n):
            for _ in range(n):
                operations += 1
    return operations


**Run the experiment.** Advance the algorithm and collect the state changes that make the behavior visible.


In [ ]:
for n in [3, 5, 10]:
    print(
        f"n={n:>2} | one loop={count_one_loop(n):>4} | "
        f"nested={count_nested_loops(n):>4} | triple={count_triple_loops(n):>4}"
    )

print("\nWhen n doubles from 5 to 10:")

print("one loop roughly doubles; nested roughly quadruples; triple roughly grows eightfold.")


## 8. Snapshots: Make State Replayable

A snapshot captures one meaningful moment in an algorithm. The later notebooks use step objects so you can replay a search, ranking process, clustering round, or matrix update.

Useful snapshots usually record:

- the step number
- the important state
- what changed
- what should still be true

In [8]:
@dataclass
class MiniStep:
    round_number: int
    score: float
    note: str


score = 1.0
steps = []
for round_number in range(1, 5):
    score = score * 0.5 + 1
    steps.append(MiniStep(round_number, score, "score = score * 0.5 + 1"))

for step in steps:
    print(f"Round {step.round_number}: score={step.score:.3f} | {step.note}")

Round 1: score=1.500 | score = score * 0.5 + 1
Round 2: score=1.750 | score = score * 0.5 + 1
Round 3: score=1.875 | score = score * 0.5 + 1
Round 4: score=1.938 | score = score * 0.5 + 1


## 9. Where This Notebook Points

Keep this map nearby as the course opens up.

| Notebook | Main idea | Key structures | Typical complexity |
|---|---|---|---|
| Dijkstra | shortest paths from one start | graph, dict, set, heap | `O((V + E) log V)` with a heap |
| A* | directed shortest path search | graph/grid, heap, heuristic | worst case like Dijkstra; often explores less |
| PageRank | repeated rank updates | graph, dict, snapshots | `O(rounds * E)` |
| Topological Sort | valid dependency order | graph, in-degree dict, queue | `O(V + E)` |
| K-means | cluster by nearest centers | points, centers, assignments | `O(rounds * k * n * dimensions)` |
| Floyd-Warshall | all-pairs shortest paths | matrix, triple loop, DP | `O(V^3)` time, `O(V^2)` memory |

`V` means vertices, `E` means edges, `n` means items, and `k` usually means clusters or choices.

## 10. The Five Questions

Use these on every algorithm in the course:

1. What are the objects?
2. What structure holds the active work?
3. What state changes each step?
4. Which loop controls the runtime?
5. What makes the algorithm stop?

<details>
<summary>Two quick answers</summary>

Dijkstra's active work is a heap frontier. Topological Sort's active work is a ready queue. Floyd-Warshall's controlling loop is the triple loop over via, start, and end.

</details>

In [9]:
def algorithm_card(name: str, active_work: str, state: str, runtime: str) -> None:
    print(name)
    print("  active work:", active_work)
    print("  state      :", state)
    print("  runtime    :", runtime)
    print()


algorithm_card(
    "Topological Sort",
    "queue of tasks with no remaining prerequisites",
    "in-degree counts and schedule",
    "O(V + E)",
)

algorithm_card(
    "Floyd-Warshall",
    "each possible middle place",
    "distance matrix",
    "O(V^3)",
)

Topological Sort
  active work: queue of tasks with no remaining prerequisites
  state      : in-degree counts and schedule
  runtime    : O(V + E)

Floyd-Warshall
  active work: each possible middle place
  state      : distance matrix
  runtime    : O(V^3)



## Visual Trace + Rigor Studio

**Problem frame.** Build the habit of tracing state and predicting growth before measuring it.

**Interactive animation target.** Plot input size against operation count and let students scrub through growth classes.

**Correctness handle.** The cost model must count the same operation under the same assumptions for every input size.

**Complexity handle.** Separate constants, lower-order terms, and asymptotic growth.

**Failure mode to test.** Wall-clock timing can lie when setup cost, cache behavior, or random input shape dominates.

**Studio task.** Measure two algorithms on the same input family, then explain where the empirical curve stops matching the asymptotic story.

**Run the experiment.** Advance the algorithm and collect the state changes that make the behavior visible.


In [ ]:
from html import escape

from pathlib import Path

import sys

from IPython.display import HTML, display

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table


**Builder helper.** Define `build_sieve_trace`, which prepares reusable examples or traces.


In [ ]:
def build_sieve_trace(limit: int = 30) -> AlgorithmTrace:
    possible = [True] * (limit + 1)
    possible[0] = False
    possible[1] = False

    trace = AlgorithmTrace(
        "Sieve of Eratosthenes",
        objective="Find primes by repeatedly crossing out known composites.",
        complexity="O(n log log n) time for the classic sieve; O(n) memory.",
    )
    trace.append(
        "initialize",
        {"possible": tuple(possible)},
        "Start with every number from 2 upward still possible.",
        metrics={"possible": sum(possible)},
        invariant="Crossed-out numbers are known not to be prime.",
        operation="initialize table",
    )

    largest_candidate = int(limit ** 0.5)
    for candidate_number in range(2, largest_candidate + 1):
        if not possible[candidate_number]:
            continue

        trace.append(
            f"select {candidate_number}",
            {"possible": tuple(possible)},
            f"{candidate_number} is still possible, so it becomes the next prime base.",
            focus={"number": candidate_number},
            metrics={"possible": sum(possible)},
            invariant="No smaller prime can divide any remaining unmarked base.",
            operation="select base",
        )

        marked = []
        for multiple in range(candidate_number * candidate_number, limit + 1, candidate_number):
            if possible[multiple]:
                marked.append(multiple)
            possible[multiple] = False

        trace.append(
            f"mark multiples of {candidate_number}",
            {"possible": tuple(possible)},
            f"Cross out multiples of {candidate_number}, starting at {candidate_number * candidate_number}.",
            focus={"number": candidate_number, "marked": tuple(marked)},
            metrics={"marked": len(marked), "possible": sum(possible)},
            invariant="Every crossed-out value has a known prime factor.",
            operation="mark multiples",
        )

    primes = tuple(number for number, is_possible in enumerate(possible) if is_possible)
    trace.append(
        "done",
        {"possible": tuple(possible), "primes": primes},
        f"The remaining possible numbers are prime: {primes}.",
        metrics={"primes": len(primes)},
        invariant="Any composite up to the limit has a prime factor no larger than sqrt(limit).",
        operation="read result",
    )
    return trace


**Visual helper.** Define `render_sieve_state`, which turns state into something students can inspect.


In [ ]:
def render_sieve_state(step: TraceStep) -> HTML:
    possible = list(step.state["possible"])
    active_number = step.focus.get("number")
    marked = set(step.focus.get("marked", ()))
    cells = []

    for cell_index, is_possible in enumerate(possible):
        if cell_index < 2:
            background = "#f1f3f5"
            color = "#8a8f98"
            label = str(cell_index)
        elif cell_index == active_number:
            background = "#f4a261"
            color = "#111111"
            label = str(cell_index)
        elif cell_index in marked:
            background = "#e76f51"
            color = "white"
            label = f"<s>{cell_index}</s>"
        elif is_possible:
            background = "#2a9d8f"
            color = "white"
            label = str(cell_index)
        else:
            background = "#e9ecef"
            color = "#8a8f98"
            label = f"<s>{cell_index}</s>"

        cells.append(
            "<span style='display:inline-flex;align-items:center;justify-content:center;"
            "width:2.1rem;height:2.1rem;margin:0.12rem;border-radius:0.25rem;"
            f"background:{background};color:{color};font-family:monospace;font-weight:700;'>"
            f"{label}</span>"
        )

    metrics = ", ".join(f"{escape(str(key))}: {escape(str(value))}" for key, value in step.metrics.items())
    return HTML(
        "<div style='line-height:1.4'>"
        f"<h4 style='margin:0 0 0.35rem 0'>{escape(step.label)}</h4>"
        f"<p style='margin:0 0 0.5rem 0'>{escape(step.description)}</p>"
        f"<div style='margin:0.5rem 0'>{''.join(cells)}</div>"
        f"<p style='margin:0.5rem 0 0 0'><strong>Invariant:</strong> {escape(step.invariant)}</p>"
        f"<p style='margin:0.25rem 0 0 0'><strong>Metrics:</strong> {metrics}</p>"
        "</div>"
    )


**Inspect the result.** Render the current state so the algorithm is easier to reason about.


In [ ]:
sieve_trace = build_sieve_trace(limit=30)

display(render_trace_table(sieve_trace, max_rows=10))

AlgorithmPlayer(sieve_trace, render_sieve_state).display()
